# CDR Dataset

This notebook explores the CDR Relation Extraction dataset from https://doi.org/10.1093/database/baw068.

**Note**: This dataset is sourced from the BioC website (https://bioc.sourceforge.net/, alternatively from https://github.com/JHnlp/BioCreative-V-CDR-Corpus) as the source linked in the paper is no longer available.

In [2]:
import json

from bioc import biocxml, biocjson
from pprint import pprint

Load and check out the dataset structure.

In [3]:
def load_file(path):
    with open(path, 'r') as fp:
        collection = biocxml.load(fp)
    
    json_string = biocjson.dumps(collection)
    return json.loads(json_string)

In [4]:
train_path = "/home/bt19d200/Ayaan/raw-datasets/BC5CDR/CDR_Data/CDR.Corpus.v010516/CDR_TrainingSet.BioC.xml"
val_path = "/home/bt19d200/Ayaan/raw-datasets/BC5CDR/CDR_Data/CDR.Corpus.v010516/CDR_DevelopmentSet.BioC.xml"
test_path = "/home/bt19d200/Ayaan/raw-datasets/BC5CDR/CDR_Data/CDR.Corpus.v010516/CDR_TestSet.BioC.xml"

ds_train = load_file(train_path)
ds_val = load_file(val_path)
ds_test = load_file(test_path)

In [5]:
print("Dataset keys:", ds_train.keys())
for key, val in ds_train.items():
    print(f"{key} type: {type(val)}")

Dataset keys: dict_keys(['bioctype', 'source', 'date', 'key', 'version', 'infons', 'documents'])
bioctype type: <class 'str'>
source type: <class 'str'>
date type: <class 'str'>
key type: <class 'str'>
version type: <class 'str'>
infons type: <class 'dict'>
documents type: <class 'list'>


In [6]:
print(ds_test['infons'].keys())

dict_keys([])


The 'documents' key contains the data.

#### Explore the dataset

In [7]:
ds_train = ds_train['documents']
ds_val = ds_val['documents']
ds_test = ds_test['documents']

In [8]:
print("Number of train samples:", len(ds_train))
print("Number of validation samples:", len(ds_val))
print("Number of test samples:", len(ds_test))

Number of train samples: 500
Number of validation samples: 500
Number of test samples: 500


In [9]:
example = ds_train[0]

pprint(example)

{'annotations': [],
 'bioctype': 'BioCDocument',
 'id': '227508',
 'infons': {},
 'passages': [{'annotations': [{'id': '0',
                                'infons': {'MESH': 'D009270',
                                           'type': 'Chemical'},
                                'locations': [{'length': 8, 'offset': 0}],
                                'text': 'Naloxone'},
                               {'id': '1',
                                'infons': {'MESH': 'D003000',
                                           'type': 'Chemical'},
                                'locations': [{'length': 9, 'offset': 49}],
                                'text': 'clonidine'}],
               'bioctype': 'BioCPassage',
               'infons': {'type': 'title'},
               'offset': 0,
               'relations': [],
               'sentences': [],
               'text': 'Naloxone reverses the antihypertensive effect of '
                       'clonidine.'},
              {'annotations': [

Check if offsets include space after title

In [13]:
def check_space(example):
    text = example['passages'][0]['text'] + ' ' + example['passages'][1]['text']
    for ann in example['passages'][1]['annotations']:
        if len(ann['locations']) == 1:
            entity = example['passages'][1]['annotations'][0]
    
    start = entity['locations'][0]['offset']
    end = start + entity['locations'][0]['length']
    if text[start:end] != entity['text']:
        print("Excludes space")
    else:
        print("Includes space")

In [14]:
check_space(example)

Includes space


Get number of entity repeats (mentions and offsets).

In [10]:
def ent_repeats(dataset):
    max_repeats, max_offsets = 1, 1
    max_offset_index, max_repeats_index = None, None
    for i, example in enumerate(dataset):
        passages = example['passages']
        entities = passages[0]['annotations'] + passages[1]['annotations']
        ent_repeat = {}
        for ent in entities:
            ent_id = ent['infons']['MESH']
            ent_repeat[ent_id] = ent_repeat[ent_id] + 1 if ent_id in ent_repeat else 1
            offsets = len(ent['locations'])
            if offsets > max_offsets:
                max_offsets = offsets
                max_offset_index = i
            
        max_repeat_ex = max(ent_repeat.values())
        if max_repeats < max_repeat_ex:
            max_repeats = max_repeat_ex
            max_repeats_index = i
    
    return {'max_offsets': {'count': max_offsets, 'index': max_offset_index}, 
            'max_repeats': {'count': max_repeats, 'index': max_repeats_index}
    }

In [11]:
print("Entity Quirks:")
print()
print("Train:")
pprint(ent_repeats(ds_train))
print()
print("Validation:")
pprint(ent_repeats(ds_val))
print()
print("Test:")
pprint(ent_repeats(ds_test))

Entity Quirks:

Train:
{'max_offsets': {'count': 2, 'index': 8},
 'max_repeats': {'count': 29, 'index': 402}}

Validation:
{'max_offsets': {'count': 2, 'index': 6},
 'max_repeats': {'count': 17, 'index': 120}}

Test:
{'max_offsets': {'count': 2, 'index': 19},
 'max_repeats': {'count': 22, 'index': 46}}


General-domain datasets have single mentions/spans per entity. Therefore, we cannot directly apply end-to-end Relation Extraction on this dataset.

In [12]:
pprint(ds_train[8])

{'annotations': [],
 'bioctype': 'BioCDocument',
 'id': '2234245',
 'infons': {},
 'passages': [{'annotations': [{'id': '0',
                                'infons': {'CompositeRole': 'CompositeMention',
                                           'MESH': 'D014786|D006311',
                                           'type': 'Disease'},
                                'locations': [{'length': 28, 'offset': 0}],
                                'text': 'Ocular and auditory toxicity'},
                               {'id': '1',
                                'infons': {'CompositeRole': 'IndividualMention',
                                           'MESH': 'D014786',
                                           'type': 'Disease'},
                                'locations': [{'length': 6, 'offset': 0},
                                              {'length': 8, 'offset': 20}],
                                'text': 'Ocular toxicity'},
                               {'id': '2',
           

Looking at the example, we can observe the following:
- Some entities are spread over two disjoint locations
- Some have 'CompositeRole' in their 'infons' while others do not
- Some have 'IndividualMention' as their 'CompositeRole' while others have 'CompositeMention'
- There are multiple instances having overlapping offsets
- The same entity occurs multiple times in a document, sometimes with different textual representations

Every entity span with a 'CompositeRole' or with two MESH IDs seperated by a '|' is a dual entity, i.e., these spans include two distinct entities clubbed together. We have to strategise on how to incorporate this into the train, validation, and test data.

The same entity having multiple textual representations make it difficult to use this dataset with end-to-end RE models operating at a mention-level.

Check for composite entities in relations

In [15]:
def composite_included(dataset):
    for example in dataset:
        for rel in example['relations']:
            if '|' in rel['infons']['Chemical'] or '|' in rel['infons']['Disease']:
                print("Composite entity found in a relation")
                return
    
    print("No composite entities found in relations")

In [16]:
print("Composite entity presence:")
print()
print("Train:")
composite_included(ds_train)
print()
print("Val:")
composite_included(ds_val)
print()
print("Test:")
composite_included(ds_test)

Composite entity presence:

Train:
No composite entities found in relations

Val:
No composite entities found in relations

Test:
No composite entities found in relations


In [18]:
pprint(ds_val[60])

{'annotations': [],
 'bioctype': 'BioCDocument',
 'id': '9022662',
 'infons': {},
 'passages': [{'annotations': [{'id': '0',
                                'infons': {'MESH': 'D010389',
                                           'type': 'Chemical'},
                                'locations': [{'length': 8, 'offset': 0}],
                                'text': 'Pemoline'},
                               {'id': '1',
                                'infons': {'MESH': 'D002819|D001264',
                                           'type': 'Disease'},
                                'locations': [{'length': 15, 'offset': 23}],
                                'text': 'choreoathetosis'}],
               'bioctype': 'BioCPassage',
               'infons': {'type': 'title'},
               'offset': 0,
               'relations': [],
               'sentences': [],
               'text': 'Pemoline induced acute choreoathetosis: case report '
                       'and review of the literatur

Solution:
- Ignore all entities that are 'CompositeMention'
- Take the first MeSH ID for entities that are not 'CompositeMention' but has two MeSH IDs (as the second ID is repeated later)
- Take the entire span containing an entity even if the entity is disjoint
- For extraction models, include all occurrences in train and validation splits
- For generative models, include all unique textual occurrences in train and validation splits
- For test split, allow selection from multiple offsets/texts for each unique ID

In [17]:
def label_statistics(ds):
    ent_types = {}
    rel_types = {}
    for ex in ds:
        passages = ex['passages']
        entities = passages[0]['annotations'] + passages[1]['annotations']
        for ent in entities:
            ent_type = ent['infons']['type']
            ent_types[ent_type] = ent_types[ent_type] + 1 if ent_type in ent_types else 1
        
        for rel in ex['relations']:
            rel_type = rel['infons']['relation']
            rel_types[rel_type] = rel_types[rel_type] + 1 if rel_type in rel_types else 1
    
    return {'ent_stats': ent_types, 'rel_stats': rel_types}

In [18]:
print("--- Dataset Label Statistics ---")
print("Train:")
pprint(label_statistics(ds_train))
print()
print("Validation:")
pprint(label_statistics(ds_val))
print()
print("Test:")
pprint(label_statistics(ds_test))

--- Dataset Label Statistics ---
Train:
{'ent_stats': {'Chemical': 5207, 'Disease': 4363}, 'rel_stats': {'CID': 1038}}

Validation:
{'ent_stats': {'Chemical': 5352, 'Disease': 4421}, 'rel_stats': {'CID': 1012}}

Test:
{'ent_stats': {'Chemical': 5394, 'Disease': 4534}, 'rel_stats': {'CID': 1066}}
